In [ ]:
#Mount drive to the code
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

In [ ]:
file_path = '/content/drive/MyDrive/Volunteership/dataset/'  

try:
  df_data = pd.read_csv(file_path + "dat.csv")
  df_md = pd.read_csv(file_path + "dat_md.csv")
except FileNotFoundError:
  print(f"Error: File not found at {file_path}")

df_data.head()
df_md.head()

In [ ]:
medications_grouped = df_md.groupby('inpatient.number')['Drug_name'].apply(list).reset_index()
merged_data = pd.merge(df_data, medications_grouped, on='inpatient.number', how='left')
merged_data = merged_data.drop_duplicates(subset='inpatient.number')
merged_data.head()

#This is the dataset obtained by merging dat.csv with dat_md.csv
data = merged_data
data.head()

In [ ]:
irr_columns = ['Unnamed: 0', 
                  'admission.ward', 
                  'admission.way', 
                  'occupation', 
                  'discharge.department', 
                  'death.within.28.days',
                  'death.within.3.months',
                  'death.within.6.months',
                  'time.of.death..days.from.admission.',
                  're.admission.time..days.from.admission.',
                  'time.to.emergency.department.within.6.months',
                  'dischargeDay']
data = data.drop(columns=irr_columns)

In [ ]:

#This is the function that accepts pandas dataframe. 
#It calculates the percentages of null values of each column, visualize in a graph and return results as a tuple.

def calculate_null_percentage(data):

  null_percentages = {col: data[col].isnull().sum() / data.shape[0] * 100 for col in data.columns}

  # Create a dataframe to display the percentage of null values for each column
  null_values_percentage = pd.DataFrame.from_dict(null_percentages, orient='index', columns=['Null Percentage'])

  # Plot the null percentages as a bar plot
  null_percentages_series = pd.Series(null_percentages).sort_values()
  plt.figure(figsize=(12, 20))
  null_percentages_series.plot(kind='barh', fontsize=8)
  plt.title('Percentage of Null Values per Column')
  plt.xlabel('Columns')
  plt.ylabel('Percentage of Null Values')
  plt.savefig('null_values_percentage.png')
  return [null_percentages, null_percentages_series]


# Calculate the percentage of null values for each column
(null_percentages, null_percentages_series) = calculate_null_percentage(data)

In [ ]:
# Truncating the columns with NA values >=50
data = data.drop(columns=null_percentages_series[null_percentages_series >= 50].index)

In [ ]:
#Performing Label Encoding
#Defining Size mapping for ordinal data 
size_mapping_for_ageCat = {'(21,29]': 1, '(29,39]': 2, '(39,49]': 3, '(49,59]': 4, '(59,69]': 5, '(69,79]': 6, '(79,89]': 7, '(89,110]': 8}
size_mapping_for_NYHA = {'I': 1, 'II': 2, 'III': 3, 'IV':4}
size_mapping_for_killip = {'I': 1, 'II': 2, 'III': 3, 'IV':4}
size_mapping_for_consciouness = {'Clear': 1, 'ResponsiveToSound': 2, 'ResponsiveToPain': 3, 'Nonresponsive': 4}

#Dictionary to map each column to its respective mapping
mapping_dict = {
    'ageCat': size_mapping_for_ageCat,
    'NYHA.cardiac.function.classification': size_mapping_for_NYHA,
    'Killip.grade': size_mapping_for_killip,
    'consciousness': size_mapping_for_consciouness
}

#Performing Label Encoding
for col in mapping_dict.keys():
    data[col] = data[col].map(mapping_dict[col])

In [ ]:
#Performing OneHot encoding

#List of columns for non-ordinal data
one_hot_columns = ['gender','type.of.heart.failure','type.II.respiratory.failure','oxygen.inhalation','outcome.during.hospitalization']

#Performing One-Hot Encoding and dropping first column to avoid dummy variable trap
data = pd.get_dummies(data, columns=one_hot_columns, drop_first=True, dtype=int)